In [ ]:
import os
import math
import argparse
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

In [ ]:
# ----------------------------- Hyperparameters -----------------------------
parser = argparse.ArgumentParser(description='PyTorch VAE on MNIST')
parser.add_argument('--batch-size', type=int, default=128)
parser.add_argument('--epochs', type=int, default=15)
parser.add_argument('--lr', type=float, default=1e-3)
parser.add_argument('--latent-dim', type=int, default=20)
parser.add_argument('--hidden-dim', type=int, default=400)
parser.add_argument('--no-cuda', action='store_true', default=False)
parser.add_argument('--seed', type=int, default=42)
parser.add_argument('--log-interval', type=int, default=200)
parser.add_argument('--output-dir', type=str, default='outputs')
args = parser.parse_args([])  # empty list so code runs inside notebook-style env

# For users running as script, uncomment the line below and comment previous one:
# args = parser.parse_args()

use_cuda = not args.no_cuda and torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')

torch.manual_seed(args.seed)

os.makedirs(args.output_dir, exist_ok=True)

In [ ]:

# ----------------------------- Data pipeline --------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=use_cuda)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=use_cuda)


In [ ]:

# ----------------------------- VAE model ------------------------------------
class VAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # Decoder
        self.fc_dec1 = nn.Linear(latent_dim, hidden_dim)
        self.fc_dec2 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h1 = F.relu(self.fc1(x))
        mu = self.fc_mu(h1)
        logvar = self.fc_logvar(h1)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        # reparameterization trick
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h3 = F.relu(self.fc_dec1(z))
        return torch.sigmoid(self.fc_dec2(h3))  # output in [0,1] for BCE

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [1]:
"""
Variational Autoencoder (VAE) - PyTorch implementation

What this file does:
- Implements a simple VAE (fully-connected) for MNIST (28x28 grayscale)
- Trains the model with ELBO = BCE reconstruction + KL divergence
- Saves generated samples and a model checkpoint

How to run:
    pip install torch torchvision matplotlib
    python vae_mnist.py

Notes:
- Uses Bernoulli likelihood (sigmoid output + binary cross-entropy).
- Device-aware (CUDA if available).
- Simple architecture, easy to extend to conv VAE or different datasets.
"""






# ----------------------------- Loss function --------------------------------
# ELBO = E_q[log p(x|z)] - KL(q(z|x) || p(z))
# For Bernoulli likelihood, use binary cross-entropy for reconstruction term.

def loss_function(recon_x, x, mu, logvar):
    # recon_x: [batch, D], x: [batch, D]
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')

    # KL divergence between q(z|x) = N(mu, sigma^2) and p(z)=N(0,I)
    # KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return BCE + KLD, BCE, KLD

# ----------------------------- Training / Evaluation ------------------------

input_dim = 28 * 28
model = VAE(input_dim=input_dim, hidden_dim=args.hidden_dim, latent_dim=args.latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=args.lr)


def train(epoch):
    model.train()
    train_loss = 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.view(-1, 28*28).to(device)
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss, bce, kld = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

        if batch_idx % args.log_interval == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}]\tLoss: {loss.item() / len(data):.4f} (BCE {bce.item()/len(data):.4f}, KLD {kld.item()/len(data):.4f})')

    avg_loss = train_loss / len(train_loader.dataset)
    print(f'====> Epoch: {epoch} Average loss: {avg_loss:.4f}')
    return avg_loss


def test(epoch):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for i, (data, _) in enumerate(test_loader):
            data = data.view(-1, 28*28).to(device)
            recon, mu, logvar = model(data)
            loss, bce, kld = loss_function(recon, data, mu, logvar)
            test_loss += loss.item()

            # save the first batch's reconstructions
            if i == 0:
                n = min(data.size(0), 8)
                comparison = torch.cat([data[:n].view(n, 1, 28, 28), recon[:n].view(n, 1, 28, 28)])
                utils.save_image(comparison.cpu(), os.path.join(args.output_dir, f'reconstruction_{epoch}.png'), nrow=n)

    avg_loss = test_loss / len(test_loader.dataset)
    print(f'====> Test set loss: {avg_loss:.4f}')
    return avg_loss


def sample_and_save(epoch, num_samples=64):
    model.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, args.latent_dim).to(device)
        samples = model.decode(z).view(-1, 1, 28, 28)
        utils.save_image(samples.cpu(), os.path.join(args.output_dir, f'samples_{epoch}.png'), nrow=8)



100%|██████████| 9.91M/9.91M [00:00<00:00, 11.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 338kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.11MB/s]


Train Epoch: 1 [0/60000]	Loss: 547.7181 (BCE 547.6080, KLD 0.1102)
Train Epoch: 1 [25600/60000]	Loss: 153.1178 (BCE 137.9201, KLD 15.1977)
Train Epoch: 1 [51200/60000]	Loss: 123.4127 (BCE 103.6798, KLD 19.7329)
====> Epoch: 1 Average loss: 165.6817
====> Test set loss: 128.3517
Saved best model (test loss 128.3517)
Train Epoch: 2 [0/60000]	Loss: 137.3884 (BCE 115.9322, KLD 21.4562)
Train Epoch: 2 [25600/60000]	Loss: 125.6231 (BCE 102.4794, KLD 23.1437)
Train Epoch: 2 [51200/60000]	Loss: 121.4261 (BCE 97.3024, KLD 24.1237)
====> Epoch: 2 Average loss: 121.8164
====> Test set loss: 115.9748
Saved best model (test loss 115.9748)
Train Epoch: 3 [0/60000]	Loss: 113.2590 (BCE 89.4637, KLD 23.7953)
Train Epoch: 3 [25600/60000]	Loss: 109.5504 (BCE 85.1501, KLD 24.4003)
Train Epoch: 3 [51200/60000]	Loss: 109.8067 (BCE 85.7856, KLD 24.0210)
====> Epoch: 3 Average loss: 114.8953
====> Test set loss: 112.4141
Saved best model (test loss 112.4141)
Train Epoch: 4 [0/60000]	Loss: 113.9137 (BCE 88.612

In [ ]:
# ----------------------------- Run training --------------------------------
if __name__ == '__main__':
    best_test = float('inf')
    for epoch in range(1, args.epochs + 1):
        train(epoch)
        test_loss = test(epoch)
        sample_and_save(epoch)

        # save best model
        if test_loss < best_test:
            best_test = test_loss
            torch.save(model.state_dict(), os.path.join(args.output_dir, 'vae_best.pth'))
            print(f'Saved best model (test loss {best_test:.4f})')

    print('Training complete. Outputs written to', args.output_dir)
